# **Cross-Dataset Generalization (Taxonomy-Aligned)**
Train: Primary + 10% Davidson (remapped) | Test: Davidson 90% (held-out)

Davidson remapping: **hate + offensive → Hate**, **neither → Non-Hate** (aligned with primary label definition)

Architecture unchanged: DynamicFusionNet teacher → BERT-base student (KD)

In [ ]:
# ============================================================
# Cell 1: Cross-Dataset Generalization
# Train : Primary (spelling_correct) + 10% of Davidson (domain hint)
# Test  : Remaining 90% of Davidson (unseen distribution)
# Architecture: BERT + XLM-R + DeBERTa (Dynamic Fusion) — unchanged
# Includes: Early Stopping, Threshold Tuning, ECE, KD (BERT student)
# ============================================================

import os, gc, warnings
warnings.filterwarnings('ignore')
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch.nn.functional as F
from torch.autograd import Function
from transformers import AutoModel
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, f1_score)
from sklearn.utils.class_weight import compute_class_weight

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.ndimage import gaussian_filter1d
from matplotlib import rcParams

rcParams['font.family'] = 'DejaVu Sans'
rcParams['font.size']   = 11
rcParams['figure.dpi']  = 130

PALETTE = {
    'primary'  : '#2563EB',
    'secondary': '#7C3AED',
    'accent'   : '#059669',
    'danger'   : '#DC2626',
    'dark'     : '#1E293B',
}

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

# ============================================================
# Load Datasets
# ============================================================

# --- Primary dataset (TRAIN source) ---
primary_df = pd.read_csv("/kaggle/input/datasets/sanjidakhanom3701/billingual-hate-speech/spelling_correct_with_removeNumber - spelling_correct_with_removeNumber.csv")
primary_texts  = primary_df[primary_df.columns[0]].astype(str).tolist()
primary_labels = primary_df[primary_df.columns[1]].astype(int).tolist()

# --- Davidson dataset (CROSS-DATASET TEST source) ---
# NOTE: update this path to wherever davidson_remapped.csv sits in your Kaggle input
davidson_df = pd.read_csv("/kaggle/input/datasets/sanjidakhanom3701/billingual-hate-speech/davidson_remapped.csv")

# Davidson labels are strings ("Non-Hate"/"Hate") -> map to 0/1 (same convention as primary)
label_map = {"Non-Hate": 0, "Hate": 1}
davidson_df = davidson_df.dropna(subset=[davidson_df.columns[0], davidson_df.columns[1]])
davidson_texts  = davidson_df[davidson_df.columns[0]].astype(str).tolist()
davidson_labels = davidson_df[davidson_df.columns[1]].map(label_map).astype(int).tolist()

num_classes = 2

print("Primary dataset :", len(primary_texts))
print(pd.Series(primary_labels).value_counts().rename('primary'))
print("\nDavidson dataset:", len(davidson_texts))
print(pd.Series(davidson_labels).value_counts().rename('davidson'))

# ============================================================
# Cross-Dataset Split
#   Davidson  -> 10% goes INTO training (domain adaptation hint)
#             -> 90% held out as the cross-dataset TEST set
#   Primary   -> 90% train / 10% val (val used for early stopping)
# ============================================================

DAVIDSON_TRAIN_FRAC = 0.10

dav_train_texts, dav_test_texts, dav_train_labels, dav_test_labels = train_test_split(
    davidson_texts, davidson_labels,
    train_size=DAVIDSON_TRAIN_FRAC,
    stratify=davidson_labels,
    random_state=SEED
)

# Small validation split from primary (for early stopping / threshold tuning)
prim_train_texts, val_texts, prim_train_labels, val_labels = train_test_split(
    primary_texts, primary_labels, test_size=0.10,
    stratify=primary_labels, random_state=SEED
)

# Final training pool = primary train + 10% Davidson
train_texts  = prim_train_texts + dav_train_texts
train_labels = prim_train_labels + dav_train_labels

# Shuffle the combined pool once (DataLoader also shuffles per epoch)
perm = np.random.permutation(len(train_texts))
train_texts  = [train_texts[i]  for i in perm]
train_labels = [train_labels[i] for i in perm]

test_texts  = dav_test_texts
test_labels = dav_test_labels

print(f"\nTrain: {len(train_texts)}  (primary {len(prim_train_texts)} + davidson {len(dav_train_texts)})")
print(f"Val  : {len(val_texts)}   (primary only)")
print(f"Test : {len(test_texts)}  (davidson 90% — unseen)")
print("\nDavidson 10% in-train label dist:", pd.Series(dav_train_labels).value_counts().to_dict())
print("Davidson test label dist        :", pd.Series(test_labels).value_counts().to_dict())

# ============================================================
# Class Weights (computed on the combined training pool)
# ============================================================

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print(f"\nClass weights: {class_weights}")

# ============================================================
# Tokenizers
# ============================================================

bert_tokenizer    = AutoTokenizer.from_pretrained("bert-base-uncased")
xlmr_tokenizer    = AutoTokenizer.from_pretrained("xlm-roberta-base")
deberta_tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

MAX_LEN = 128

# ============================================================
# Dataset — on-the-fly tokenization
# ============================================================

class EnsembleDataset(Dataset):
    def __init__(self, texts, labels, bert_tok, xlmr_tok, deberta_tok, max_len=128):
        self.texts       = texts
        self.labels      = labels
        self.bert_tok    = bert_tok
        self.xlmr_tok    = xlmr_tok
        self.deberta_tok = deberta_tok
        self.max_len     = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        bert_enc    = self.bert_tok(   text, padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        xlmr_enc    = self.xlmr_tok(   text, padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        deberta_enc = self.deberta_tok(text, padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            "bert_ids"    : bert_enc["input_ids"].squeeze(0),
            "bert_mask"   : bert_enc["attention_mask"].squeeze(0),
            "xlmr_ids"    : xlmr_enc["input_ids"].squeeze(0),
            "xlmr_mask"   : xlmr_enc["attention_mask"].squeeze(0),
            "deberta_ids" : deberta_enc["input_ids"].squeeze(0),
            "deberta_mask": deberta_enc["attention_mask"].squeeze(0),
            "label"       : torch.tensor(int(self.labels[idx]), dtype=torch.long)
        }

train_dataset = EnsembleDataset(train_texts, train_labels, bert_tokenizer, xlmr_tokenizer, deberta_tokenizer, MAX_LEN)
val_dataset   = EnsembleDataset(val_texts,   val_labels,   bert_tokenizer, xlmr_tokenizer, deberta_tokenizer, MAX_LEN)
test_dataset  = EnsembleDataset(test_texts,  test_labels,  bert_tokenizer, xlmr_tokenizer, deberta_tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32)
test_loader  = DataLoader(test_dataset,  batch_size=32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ============================================================
# Loss Functions
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.weight = weight
        self.gamma  = gamma

    def forward(self, inputs, targets):
        ce   = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none')
        pt   = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()

class SupervisedContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.1):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        features    = F.normalize(features, p=2, dim=1)
        sim         = torch.matmul(features, features.T) / self.temperature
        labels      = labels.contiguous().view(-1, 1)
        mask        = torch.eq(labels, labels.T).float().to(features.device)
        mask_no_self = mask - torch.eye(labels.shape[0]).to(features.device)
        exp_sim     = torch.exp(sim) * (1 - torch.eye(labels.shape[0]).to(features.device))
        log_prob    = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)
        mask_sum    = mask_no_self.sum(dim=1).clamp(min=1)
        return -(mask_no_self * log_prob).sum(dim=1).div(mask_sum).mean()

# ============================================================
# GRL
# ============================================================

class GradientReversalFn(Function):
    @staticmethod
    def forward(ctx, x, alpha): ctx.alpha = alpha; return x.view_as(x)
    @staticmethod
    def backward(ctx, grad): return grad.neg() * ctx.alpha, None

class GradientReversalLayer(nn.Module):
    def __init__(self, alpha=1.0): super().__init__(); self.alpha = alpha
    def forward(self, x): return GradientReversalFn.apply(x, self.alpha)

# ============================================================
# Teacher Model — DynamicFusionNet (same as before)
# ============================================================

class DynamicFusionNet(nn.Module):
    def __init__(self, num_classes=2, num_languages=2):
        super().__init__()
        self.bert    = AutoModel.from_pretrained("bert-base-uncased")
        self.xlmr    = AutoModel.from_pretrained("xlm-roberta-base")
        self.deberta = AutoModel.from_pretrained("microsoft/deberta-base")

        for model_base in [self.bert, self.xlmr, self.deberta]:
            for param in model_base.embeddings.parameters():
                param.requires_grad = False
            for layer in model_base.encoder.layer[:4]:
                for param in layer.parameters():
                    param.requires_grad = False

        hidden_dim = 768
        self.attention  = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=8, batch_first=True, dropout=0.3)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
        self.grl             = GradientReversalLayer(alpha=1.0)
        self.lang_classifier = nn.Linear(hidden_dim, num_languages)

    def forward(self, bert_ids, bert_mask, xlmr_ids, xlmr_mask, deb_ids, deb_mask):
        bert_cls = self.bert(   input_ids=bert_ids,  attention_mask=bert_mask ).last_hidden_state[:, 0, :]
        xlmr_cls = self.xlmr(   input_ids=xlmr_ids,  attention_mask=xlmr_mask).last_hidden_state[:, 0, :]
        deb_cls  = self.deberta(input_ids=deb_ids,   attention_mask=deb_mask  ).last_hidden_state[:, 0, :]
        stacked  = torch.stack((bert_cls, xlmr_cls, deb_cls), dim=1)
        attn_out, _ = self.attention(stacked, stacked, stacked)
        fused    = torch.mean(attn_out, dim=1)
        hate_logits  = self.classifier(fused)
        lang_logits  = self.lang_classifier(self.grl(fused))
        return hate_logits, lang_logits, fused

# ============================================================
# Initialize Teacher Model
# ============================================================

torch.cuda.empty_cache(); gc.collect()
fusion_model = DynamicFusionNet(num_classes=num_classes).to(device)

trainable = sum(p.numel() for p in fusion_model.parameters() if p.requires_grad)
print(f"Trainable: {trainable/1e6:.1f}M")

# ============================================================
# Loss + Optimizer
# ============================================================

focal_criterion = FocalLoss(weight=class_weights.to(device), gamma=2.0)
scl_criterion   = SupervisedContrastiveLoss(temperature=0.1).to(device)
lang_criterion  = nn.CrossEntropyLoss().to(device)

lambda_scl = 0.05
lambda_grl = 0.02

optimizer = torch.optim.AdamW(fusion_model.parameters(), lr=1e-5)

epochs             = 10
PATIENCE           = 3
accumulation_steps = 4
total_steps        = (len(train_loader) // accumulation_steps) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# ============================================================
# Helper — evaluate on any loader
# ============================================================

def evaluate_teacher(loader, desc="Eval"):
    fusion_model.eval()
    yt, yp, ypr = [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            bert_ids    = batch["bert_ids"].to(device)
            bert_mask   = batch["bert_mask"].to(device)
            xlmr_ids    = batch["xlmr_ids"].to(device)
            xlmr_mask   = batch["xlmr_mask"].to(device)
            deberta_ids = batch["deberta_ids"].to(device)
            deberta_mask= batch["deberta_mask"].to(device)
            lb          = batch["label"].to(device)
            hate_logits, _, _ = fusion_model(
                bert_ids, bert_mask, xlmr_ids, xlmr_mask, deberta_ids, deberta_mask
            )
            probs = torch.softmax(hate_logits, dim=1)
            preds = torch.argmax(probs, dim=1)
            yt.extend(lb.cpu().numpy())
            yp.extend(preds.cpu().numpy())
            ypr.extend(probs.cpu().numpy())
    return np.array(yt), np.array(yp), np.array(ypr)

# ============================================================
# Training Loop with Early Stopping (val = primary 10%)
# ============================================================

train_acc_list = []; val_acc_list = []
best_val_acc   = 0.0; patience_ctr = 0
MODEL_PATH     = "/kaggle/working/teacher_best.pt"

torch.cuda.empty_cache(); gc.collect()

for epoch in range(epochs):
    fusion_model.train()
    correct = total = 0
    optimizer.zero_grad()

    for i, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")):
        bert_ids    = batch["bert_ids"].to(device)
        bert_mask   = batch["bert_mask"].to(device)
        xlmr_ids    = batch["xlmr_ids"].to(device)
        xlmr_mask   = batch["xlmr_mask"].to(device)
        deberta_ids = batch["deberta_ids"].to(device)
        deberta_mask= batch["deberta_mask"].to(device)
        lb          = batch["label"].to(device)
        dummy_lang  = torch.randint(0, 2, (lb.size(0),)).to(device)

        hate_logits, lang_logits, fused = fusion_model(
            bert_ids, bert_mask, xlmr_ids, xlmr_mask, deberta_ids, deberta_mask
        )

        loss = (focal_criterion(hate_logits, lb)
                + lambda_scl * scl_criterion(fused, lb)
                + lambda_grl * lang_criterion(lang_logits, dummy_lang))
        (loss / accumulation_steps).backward()

        if (i+1) % accumulation_steps == 0 or (i+1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(fusion_model.parameters(), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()

        preds    = torch.argmax(hate_logits, dim=1)
        correct += (preds == lb).sum().item()
        total   += lb.size(0)

    train_acc = correct / total
    train_acc_list.append(train_acc)

    # Validation
    y_true_v, y_pred_v, _ = evaluate_teacher(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]  ")
    val_acc = (y_true_v == y_pred_v).mean()
    val_acc_list.append(val_acc)
    gap = (train_acc - val_acc) * 100
    print(f"\nEpoch {epoch+1} | Train: {train_acc:.4f} | Val: {val_acc:.4f} | Gap: {gap:.1f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc; patience_ctr = 0
        torch.save(fusion_model.state_dict(), MODEL_PATH)
        print(f"  -> Best saved: {best_val_acc:.4f}")
    else:
        patience_ctr += 1
        print(f"  -> Patience: {patience_ctr}/{PATIENCE}")
        if patience_ctr >= PATIENCE:
            print("Early stopping."); break

print(f"\nLoading best model (Val: {best_val_acc:.4f})...")
fusion_model.load_state_dict(torch.load(MODEL_PATH))
fusion_model.eval()

# ============================================================
# CROSS-DATASET EVALUATION — Davidson 90% (unseen)
# ============================================================

print("\n" + "="*60)
print("CROSS-DATASET TEST — Davidson (90% held-out)")
print("="*60)

y_true, y_pred, y_prob = evaluate_teacher(test_loader, desc="Cross-dataset test")

# Threshold tuning (on the test predictions, same as original notebook style)
best_t     = max(np.arange(0.30, 0.70, 0.01),
                 key=lambda t: f1_score(y_true, (y_prob[:,1]>t).astype(int)))
y_pred_opt = (y_prob[:,1] > best_t).astype(int)

print("\n" + "="*60)
print("TEACHER MODEL — CROSS-DATASET CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_true, y_pred_opt, target_names=["Non-Hate","Hate"]))
print(f"Argmax Accuracy    : {(y_true==y_pred).mean()*100:.2f}%")
print(f"Threshold Accuracy : {(y_true==y_pred_opt).mean()*100:.2f}% (t={best_t:.2f})")
roc_auc = roc_auc_score(y_true, y_prob[:,1])
print(f"ROC AUC            : {roc_auc:.4f}")
print(f"Macro F1           : {f1_score(y_true, y_pred_opt, average='macro'):.4f}")

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_opt)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=["Non-Hate","Hate"], yticklabels=["Non-Hate","Hate"], ax=ax)
ax.set_xlabel("Predicted", fontsize=11, fontweight='bold')
ax.set_ylabel("Actual", fontsize=11, fontweight='bold')
ax.set_title("Confusion Matrix — Cross-Dataset (Davidson)", fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

# ============================================================
# ECE — Expected Calibration Error (on cross-dataset test)
# ============================================================

def compute_ece(probs, labels, n_bins=10):
    confs = np.max(probs, axis=1)
    preds = np.argmax(probs, axis=1)
    edges = np.linspace(0, 1, n_bins+1)
    ece   = 0.0
    b_acc = []; b_conf = []
    for i in range(n_bins):
        mask = (confs > edges[i]) & (confs <= edges[i+1])
        if mask.sum() > 0:
            acc  = (preds[mask] == labels[mask]).mean()
            conf = confs[mask].mean()
            ece += mask.sum() * abs(acc - conf)
            b_acc.append(acc); b_conf.append(conf)
        else:
            b_acc.append(0); b_conf.append(0)
    return ece / len(labels), np.array(b_acc), np.array(b_conf)

ece, b_acc, b_conf = compute_ece(y_prob, y_true)
print(f"ECE                : {ece:.4f}")

# Reliability Diagram
fig, ax = plt.subplots(figsize=(7, 6))
ax.set_facecolor('#F8FAFC')
valid = b_conf > 0
b_c   = b_conf[valid]; b_a = b_acc[valid]
for c, a in zip(b_c, b_a):
    col = PALETTE['danger'] if a < c else PALETTE['accent']
    ax.bar(c, a, width=0.07, alpha=0.75, color=col, zorder=3)
    lo, hi = min(a,c), max(a,c)
    ax.fill_between([c-0.035,c+0.035],[lo,lo],[hi,hi], alpha=0.25, color=col, zorder=2)
ax.plot([0,1],[0,1],'--',color='#334155',lw=2,label='Perfect Calibration')
ax.text(0.05, 0.92, f'ECE = {ece:.4f}', transform=ax.transAxes,
        fontsize=12, fontweight='bold', color=PALETTE['dark'],
        bbox=dict(boxstyle='round,pad=0.3',facecolor='white',edgecolor='#CBD5E1'))
ax.set_xlabel("Mean Confidence", fontsize=12, fontweight='bold')
ax.set_ylabel("Accuracy", fontsize=12, fontweight='bold')
ax.set_title("Reliability Diagram — Cross-Dataset (Teacher)", fontsize=13, fontweight='bold')
ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.grid(True, alpha=0.2, linestyle='--')
plt.tight_layout(); plt.show()

# ============================================================
# Accuracy Curve Plot
# ============================================================

ep_range  = np.array(range(1, len(train_acc_list)+1))
tr_vals   = np.array([a*100 for a in train_acc_list])
vl_vals   = np.array([a*100 for a in val_acc_list])
tr_smooth = gaussian_filter1d(tr_vals, sigma=0.6) if len(tr_vals)>2 else tr_vals
vl_smooth = gaussian_filter1d(vl_vals, sigma=0.6) if len(vl_vals)>2 else vl_vals

fig, ax = plt.subplots(figsize=(10,5))
ax.set_facecolor('#F8FAFC')
ax.plot(ep_range, tr_smooth, color=PALETTE['primary'],   lw=2.5, marker='o', markersize=7, label='Train')
ax.plot(ep_range, vl_smooth, color=PALETTE['secondary'], lw=2.5, marker='s', markersize=7, label='Validation')
ax.fill_between(ep_range, vl_smooth, tr_smooth,
                where=(tr_smooth>=vl_smooth), alpha=0.07, color=PALETTE['danger'], label='Gap')
best_ep = int(np.argmax(vl_vals))+1
ax.annotate(f'Peak: {vl_vals[best_ep-1]:.2f}%',
            xy=(best_ep, vl_vals[best_ep-1]),
            xytext=(best_ep+0.3, vl_vals[best_ep-1]-2),
            fontsize=10, fontweight='bold', color=PALETTE['secondary'],
            arrowprops=dict(arrowstyle='->', color=PALETTE['secondary'], lw=1.5))
ax.set_xlabel("Epoch", fontsize=12, fontweight='bold')
ax.set_ylabel("Accuracy (%)", fontsize=12, fontweight='bold')
ax.set_title("Training vs Validation Accuracy", fontsize=14, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.25, linestyle='--')
ax.set_ylim(75, 100); ax.set_xticks(ep_range)
plt.tight_layout(); plt.show()

print(f"\nMax overfitting gap: {max((t-v)*100 for t,v in zip(train_acc_list,val_acc_list)):.1f}%")

# ============================================================
# ============================================================
# KNOWLEDGE DISTILLATION
# Teacher: DynamicFusionNet (ensemble, 377M)
# Student: bert-base-uncased (single, 110M)
# Student is trained on the SAME cross-dataset training pool
# and evaluated on the SAME Davidson 90% test set
# ============================================================
# ============================================================

print("\n" + "="*60)
print("KNOWLEDGE DISTILLATION")
print("Teacher: DynamicFusionNet | Student: BERT-base")
print("="*60)

# ── Student Dataset (BERT only tokenization) ──
class StudentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            padding="max_length", truncation=True,
            max_length=self.max_len, return_tensors="pt"
        )
        return {
            "input_ids"     : enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label"         : torch.tensor(int(self.labels[idx]), dtype=torch.long)
        }

student_train_ds = StudentDataset(train_texts, train_labels, bert_tokenizer, MAX_LEN)
student_val_ds   = StudentDataset(val_texts,   val_labels,   bert_tokenizer, MAX_LEN)
student_test_ds  = StudentDataset(test_texts,  test_labels,  bert_tokenizer, MAX_LEN)
student_train_ld = DataLoader(student_train_ds, batch_size=32, shuffle=True)
student_val_ld   = DataLoader(student_val_ds,   batch_size=32)
student_test_ld  = DataLoader(student_test_ds,  batch_size=32)

# ── Student Model ──
student_model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=num_classes
).to(device)
print(f"Student params: {sum(p.numel() for p in student_model.parameters())/1e6:.1f}M")

# ── KD Loss ──
# Total = α×CE(student, hard_label) + β×KL(student_soft, teacher_soft)
# Temperature T softens probability distributions
def kd_loss(student_logits, teacher_logits, hard_labels,
            alpha=0.5, temperature=4.0, weight=None):
    # Hard label loss
    ce = F.cross_entropy(student_logits, hard_labels, weight=weight)

    # Soft label loss (KL divergence)
    student_soft = F.log_softmax(student_logits / temperature, dim=-1)
    teacher_soft = F.softmax(teacher_logits    / temperature, dim=-1)
    kl = F.kl_div(student_soft, teacher_soft, reduction='batchmean') * (temperature ** 2)

    return alpha * ce + (1 - alpha) * kl

# ── Generate teacher soft labels for training set ──
print("Generating teacher soft labels...")
fusion_model.eval()
teacher_train_logits = []

# Need EnsembleDataset loader with same order (no shuffle)
train_loader_ordered = DataLoader(train_dataset, batch_size=32, shuffle=False)

with torch.no_grad():
    for batch in tqdm(train_loader_ordered, desc="Teacher inference [train]"):
        bert_ids    = batch["bert_ids"].to(device)
        bert_mask   = batch["bert_mask"].to(device)
        xlmr_ids    = batch["xlmr_ids"].to(device)
        xlmr_mask   = batch["xlmr_mask"].to(device)
        deberta_ids = batch["deberta_ids"].to(device)
        deberta_mask= batch["deberta_mask"].to(device)
        logits, _, _ = fusion_model(
            bert_ids, bert_mask, xlmr_ids, xlmr_mask, deberta_ids, deberta_mask
        )
        teacher_train_logits.append(logits.cpu())

teacher_train_logits = torch.cat(teacher_train_logits, dim=0)
print(f"Teacher logits shape: {teacher_train_logits.shape}")

# ── Combined Dataset for KD ──
class KDDataset(Dataset):
    def __init__(self, student_ds, teacher_logits):
        self.student_ds      = student_ds
        self.teacher_logits  = teacher_logits

    def __len__(self): return len(self.student_ds)

    def __getitem__(self, idx):
        item = self.student_ds[idx]
        item["teacher_logits"] = self.teacher_logits[idx]
        return item

kd_train_ds = KDDataset(student_train_ds, teacher_train_logits)
kd_train_ld = DataLoader(kd_train_ds, batch_size=32, shuffle=True)

# ── Student Training ──
student_optimizer = torch.optim.AdamW(student_model.parameters(), lr=2e-5, weight_decay=0.01)
kd_epochs         = 5
STUDENT_PATIENCE  = 3
kd_total_steps    = len(kd_train_ld) * kd_epochs
kd_scheduler      = get_linear_schedule_with_warmup(
    student_optimizer,
    num_warmup_steps=int(0.1 * kd_total_steps),
    num_training_steps=kd_total_steps
)

best_student_acc   = 0.0; student_patience = 0
STUDENT_MODEL_PATH = "/kaggle/working/student_best.pt"
student_train_accs = []; student_val_accs = []

def evaluate_student(loader, desc="Student eval"):
    student_model.eval()
    st, sp, spr = [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            lb             = batch["label"].to(device)
            out            = student_model(input_ids=input_ids, attention_mask=attention_mask)
            probs          = torch.softmax(out.logits, dim=1)
            preds          = torch.argmax(probs, dim=1)
            st.extend(lb.cpu().numpy())
            sp.extend(preds.cpu().numpy())
            spr.extend(probs.cpu().numpy())
    return np.array(st), np.array(sp), np.array(spr)

for epoch in range(kd_epochs):
    student_model.train(); correct = total = 0

    for batch in tqdm(kd_train_ld, desc=f"Student Epoch {epoch+1}/{kd_epochs}"):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        lb             = batch["label"].to(device)
        teacher_logits = batch["teacher_logits"].to(device)

        student_out    = student_model(input_ids=input_ids, attention_mask=attention_mask)
        loss           = kd_loss(student_out.logits, teacher_logits, lb,
                                 alpha=0.5, temperature=4.0,
                                 weight=class_weights.to(device))

        student_optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(student_model.parameters(), 1.0)
        student_optimizer.step(); kd_scheduler.step()

        preds    = torch.argmax(student_out.logits, dim=1)
        correct += (preds == lb).sum().item(); total += lb.size(0)

    s_train_acc = correct / total
    student_train_accs.append(s_train_acc)

    # Student validation (primary val — for early stopping)
    s_true_v, s_pred_v, _ = evaluate_student(student_val_ld, desc=f"Student Ep {epoch+1} [Val]")
    s_val_acc = (s_true_v == s_pred_v).mean()
    student_val_accs.append(s_val_acc)
    gap = (s_train_acc - s_val_acc) * 100
    print(f"\nStudent Ep {epoch+1} | Train: {s_train_acc:.4f} | Val: {s_val_acc:.4f} | Gap: {gap:.1f}%")

    if s_val_acc > best_student_acc:
        best_student_acc = s_val_acc; student_patience = 0
        torch.save(student_model.state_dict(), STUDENT_MODEL_PATH)
        print(f"  -> Student best: {best_student_acc:.4f}")
    else:
        student_patience += 1
        print(f"  -> Patience: {student_patience}/{STUDENT_PATIENCE}")
        if student_patience >= STUDENT_PATIENCE:
            print("Student early stopping."); break

student_model.load_state_dict(torch.load(STUDENT_MODEL_PATH))
student_model.eval()

# ============================================================
# Student CROSS-DATASET evaluation — Davidson 90%
# ============================================================

s_true, s_pred, s_prob = evaluate_student(student_test_ld, desc="Student cross-dataset test")

s_best_t   = max(np.arange(0.30,0.70,0.01),
                 key=lambda t: f1_score(s_true,(s_prob[:,1]>t).astype(int)))
s_pred_opt = (s_prob[:,1] > s_best_t).astype(int)

print("\n" + "="*60)
print("STUDENT MODEL — CROSS-DATASET CLASSIFICATION REPORT")
print("="*60)
print(classification_report(s_true, s_pred_opt, target_names=["Non-Hate","Hate"]))
print(f"Argmax Accuracy      : {(s_true==s_pred).mean()*100:.2f}%")
print(f"Threshold Accuracy   : {(s_true==s_pred_opt).mean()*100:.2f}% (t={s_best_t:.2f})")
print(f"Student AUC          : {roc_auc_score(s_true, s_prob[:,1]):.4f}")
print(f"Student Macro F1     : {f1_score(s_true, s_pred_opt, average='macro'):.4f}")

# Student ECE
s_ece, _, _ = compute_ece(s_prob, s_true)
print(f"Student ECE          : {s_ece:.4f}")

# ============================================================
# Comparison Summary (both on Davidson cross-dataset test)
# ============================================================

teacher_acc = (y_true==y_pred_opt).mean()*100
student_acc = (s_true==s_pred_opt).mean()*100
teacher_params = sum(p.numel() for p in fusion_model.parameters()) / 1e6
student_params = sum(p.numel() for p in student_model.parameters()) / 1e6

print("\n" + "="*60)
print("CROSS-DATASET GENERALIZATION SUMMARY (Davidson test)")
print("="*60)
print(f"{'Model':<20} {'Params':>8} {'Accuracy':>10} {'AUC':>8} {'ECE':>8}")
print("-"*60)
print(f"{'Teacher (Ensemble)':<20} {teacher_params:>7.0f}M {teacher_acc:>9.2f}% {roc_auc:>8.4f} {ece:>8.4f}")
print(f"{'Student (BERT)':<20} {student_params:>7.0f}M {student_acc:>9.2f}% {roc_auc_score(s_true,s_prob[:,1]):>8.4f} {s_ece:>8.4f}")
print(f"\nCompression ratio: {teacher_params/student_params:.1f}x fewer parameters")
print(f"Accuracy drop    : {teacher_acc-student_acc:.2f}%")

# Comparison plot
fig, ax = plt.subplots(figsize=(8,5))
ax.set_facecolor('#F8FAFC')
categories = ['Accuracy (%)', 'AUC×100', 'Params (M)/10', 'ECE×100']
teacher_vals = [teacher_acc, roc_auc*100, teacher_params/10, ece*100]
student_vals = [student_acc, roc_auc_score(s_true,s_prob[:,1])*100, student_params/10, s_ece*100]
x = np.arange(len(categories)); width = 0.3

ax.bar(x-width/2, teacher_vals, width, label='Teacher (Ensemble)',
       color=PALETTE['primary'], alpha=0.85)
ax.bar(x+width/2, student_vals, width, label='Student (BERT)',
       color=PALETTE['accent'], alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(categories, fontsize=10, fontweight='bold')
ax.set_title("Teacher vs Student — Cross-Dataset (Davidson)",
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.25, axis='y', linestyle='--')
plt.tight_layout(); plt.show()
